# 06. 연구 확장: 환자 교차검증과 공식 예제

기존 두 test 환자(HCC02/HCC09)는 제외하고 8명의 개발 환자만 사용합니다. 4개 outer fold에서 각 환자는 한 번만 평가됩니다. 내부 validation으로 head 설정을 고른 뒤, 해당 outer fold의 나머지 환자에 다시 학습합니다. 개발 결과를 반복 확인하므로 새로운 blind test가 아닙니다.

기본 동작은 이미 실행된 작은 결과를 읽습니다. 대규모 추출이나 전체 학습을 자동 실행하지 않습니다. 세포 분류, 공식 MS 예제, 실제 섭동 반응 예측은 서로 다른 task입니다.


In [1]:
from pathlib import Path
import json, sys, subprocess
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'configs/research/protocol_annotation.yaml').exists()
REPORT = ROOT / 'reports/research_phase2'
pd.read_csv(REPORT / 'annotation_pilot_summary/pooled_metrics.tsv', sep='\t')


,model,n_cells,pooled_macro_f1,balanced_accuracy,mean_fold_macro_f1,development_only
0,pca,8760,0.938026,0.952574,0.932742,True
1,scgpt,8760,0.940900,0.951443,0.934294,True
2,geneformer,8760,0.947063,0.946498,0.941677,True


## 어떤 세포에서 오차가 나는가?

pooled macro-F1은 fold 점수의 단순 평균과 다릅니다. 아래에는 8명 전체의 class별 support와 donor별 결과를 함께 봅니다. 특정 환자에게 없는 class도 공통 6개 class 점수에서는 0으로 처리되므로 support와 함께 해석합니다.


In [2]:
pd.read_csv(REPORT / 'annotation_pilot_summary/per_class.tsv', sep='\t')


,model,label,precision,recall,f1-score,support
0,pca,B,0.780362,0.917933,0.843575,329.0
1,pca,Endothelial,0.929435,0.954451,0.941777,483.0
2,pca,Fibroblast,0.942373,0.958621,0.950427,290.0
3,pca,Hepatocyte,0.942586,0.968180,0.955211,2357.0
4,pca,Myeloid,0.970487,0.969533,0.970010,2035.0
5,pca,T/NK,0.988491,0.946724,0.967157,3266.0
6,scgpt,B,0.835655,0.911854,0.872093,329.0
7,scgpt,Endothelial,0.922000,0.954451,0.937945,483.0
8,scgpt,Fibroblast,0.925926,0.948276,0.936968,290.0
9,scgpt,Hepatocyte,0.946866,0.967756,0.957197,2357.0


In [3]:
pd.read_csv(REPORT / 'annotation_pilot_summary/by_donor.tsv', sep='\t')


,model,patient_id,n_cells,macro_f1_common_ontology,n_errors
0,pca,HCC01,472,0.920872,20
1,pca,HCC03,1032,0.896811,56
2,pca,HCC04,958,0.952256,46
3,pca,HCC05,1113,0.953956,35
4,pca,HCC06,1219,0.906370,62
5,pca,HCC07,846,0.906563,47
6,pca,HCC08,1909,0.935904,67
7,pca,HCC10,1211,0.944844,39
8,scgpt,HCC01,472,0.872666,23
9,scgpt,HCC03,1032,0.900721,57


## 공식 scGPT MS 예제

공식 reference/query 파일, seed 0, 10 epoch, batch size 32를 유지했습니다. 현재 장비에서는 non-flash attention을 사용하며, 공식 weight-key 변환 후 attention 24개 텐서가 일치하는지 확인했습니다. 이 설정 차이 때문에 동일한 수치의 완전 재현으로 단정하지 않습니다. 64-cell smoke 점수와 전체 실행 점수를 구별합니다.


In [4]:
official = json.loads((REPORT / 'runs/R00_ms_full_seed0_v2/run.json').read_text())
print('status:', official['reproduction_status'])
print('reference/query:', official['n_reference'], official['n_evaluation'])
print('metrics:', official['metrics'])
print('training seconds:', official['wall_training_seconds'])
print('GPU GiB:', official['peak_process_gpu_gib'])


status: modified_reproduction
reference/query: 7844 13468
metrics: {'test/accuracy': 0.8635283635283635, 'test/precision': 0.7153100418139433, 'test/recall': 0.6944365520156915, 'test/macro_f1': 0.6755809331791585}
training seconds: 1476.0225419513881
GPU GiB: 9.80078125


## 원할 때 개발 fold 하나 다시 실행

아래 `RUN_ONE_FOLD`를 True로 바꾸면 이미 저장된 scGPT 파일럿 표현에 대해 CPU로 head를 다시 학습합니다. 새로운 run ID를 사용하고 기존 결과를 덮어쓰지 않습니다. 새 count/embedding 추출, 외부 cohort 평가, 4개 fold 전체 실행은 하지 않습니다.


In [5]:
RUN_ONE_FOLD = False
if RUN_ONE_FOLD:
    from datetime import datetime, timezone
    run_id = 'tutorial06_scgpt_fold0_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
    subprocess.run([sys.executable, str(ROOT / 'scripts/research/run_annotation_cv.py'),
                    '--config', str(ROOT / 'configs/research/protocol_annotation.yaml'),
                    '--model', 'scgpt', '--fold', '0', '--run-id', run_id], cwd=ROOT, check=True)
    display(pd.read_json(REPORT / 'runs' / run_id / 'metrics.json', typ='series'))
else:
    print('저장 결과 읽기 모드: 새 학습을 실행하지 않았습니다.')


저장 결과 읽기 모드: 새 학습을 실행하지 않았습니다.


## 실행 범위와 외부 평가

전체 donor CV, scVI/LiverCT 비교, 실제 GEARS 학습, 분류 미세조정과 전 fold·3개 seed의 대조를 실행했습니다. 외부 cohort는 모델·데이터 선택을 고정한 뒤 최종 추론하며, 최신 완료 상태는 08과 연구 보고서에서 확인합니다. 서로 다른 task의 점수를 하나의 순위로 합치지 않습니다.


## 전체 개발 데이터와 query adaptation 결과

63,036개 세포 모두가 한 번씩 평가됩니다. 아래 첫 표는 train-only fitting의 inductive 결과, 둘째 표는 외부 reference에 비라벨 query를 적응시킨 별도 결과입니다. Unknown을 제거한 점수만으로 비교하지 않고 prediction coverage를 함께 봅니다. LiverCT의 reference·label 양과 native head도 다릅니다.


In [6]:
display(pd.read_csv(REPORT / 'annotation_full_summary/pooled_metrics.tsv', sep='\t'))
display(pd.read_csv(REPORT / 'annotation_full_query_adaptive_summary/pooled_metrics.tsv', sep='\t'))


,model,n_cells,pooled_macro_f1,balanced_accuracy,mean_fold_macro_f1,prediction_coverage,answered_only_macro_f1,development_only
0,pca,63036,0.934829,0.946531,0.927569,1.0,0.934829,True
1,scgpt,63036,0.937697,0.945257,0.929022,1.0,0.937697,True
2,geneformer,63036,0.947083,0.946851,0.938952,1.0,0.947083,True
3,geneformer_clcancer,63036,0.945840,0.944824,0.937646,1.0,0.945840,True
4,scvi,63036,0.937858,0.939070,0.930143,1.0,0.937858,True


,model,n_cells,pooled_macro_f1,balanced_accuracy,mean_fold_macro_f1,prediction_coverage,answered_only_macro_f1,development_only
0,LiverCT_native,63036,0.901671,0.879932,0.890887,0.887096,0.945821,True
1,LiverCT_SCANVI_control,63036,0.706087,0.779307,0.700043,0.803176,0.783205,True


## 실제 LoRA 연결 확인

Geneformer의 `hidden_states[11]` CLS에 영향을 주는 encoder block 0–10의 query/value 22개 모듈을 사용했습니다. 최종 block 11은 이 표현에 영향을 주지 않으므로 제외했습니다. adapter 초기 출력 일치, 실제 gradient/update, adapter와 head 저장·복원을 확인했습니다. 연결 검증을 분류 성능 개선으로 해석하지 않습니다.


In [7]:
probe = json.loads((REPORT / 'runs/M03_lora_module_gradient_reload_probe_v1/run.json').read_text())
print('adapter/head parameters:', probe['adapter_parameter_count'], probe['head_parameter_count'])
print('initial output difference:', probe['initial_output_max_abs_difference'])
print('save/reload difference:', probe['save_reload_logits_max_abs_difference'])


adapter/head parameters: 270336 99206
initial output difference: 0.0
save/reload difference: 0.0


## 데이터만 바꾼 첫 fold 대조

개선 실험은 학습량을 4,096개로 맞춘 별도 조건입니다. 12,000개로 학습한 앞선 A-series를 대조 점수로 재사용하지 않습니다. 모델과 C/class_weight는 고정하고 donor-balanced, donor×class-balanced sampling을 비교했습니다. validation/evaluation ID 및 학습 셀 수가 동일한지 실제 manifest로 확인합니다. 한 fold 점수만으로 개선을 확정하지 않습니다.


In [8]:
display(pd.read_csv(REPORT / 'sampling_screen/metrics.tsv', sep='\t'))
print(json.loads((REPORT / 'sampling_screen/audit.json').read_text()))


,run_id,data_variant,n_outer_training,n_evaluation,inner_validation_macro_f1,development_outer_macro_f1,selected_C,selected_class_weight
0,D00_full4096_fold0_seed42_v1,D0_natural,4096,12082,0.938944,0.932355,0.01,balanced
1,D1_donor_balanced_full4096_fold0_seed42_v1,D1_donor_balanced,4096,12082,0.928110,0.916089,0.01,balanced
2,D2_donor_class_balanced_full4096_fold0_seed42_v1,D2_donor_class_balanced,4096,12082,0.918294,0.901284,0.01,balanced


{'status': 'executed', 'project_commit': '1594da0cb1eeb1241e6595ee6f9f7ce6238c4696', 'checks': ['same seed/fold/checkpoint/head hyperparameters', 'identical training cell count', 'identical validation manifest hash', 'identical ordered evaluation population', 'no outer-training/evaluation overlap', 'actual manifest hashes verified'], 'selected_data_candidate': 'D1_donor_balanced', 'selection_metric': 'inner-validation fixed-six macroF1 only', 'is_confirmed_improvement': False, 'confirmation': 'all donor folds and seeds42,43,44 still required', 'external_final_scores_observed': False}


## 비교 정보량과 작은 saved-model 추론

Inductive 모델과 query-adaptive reference mapping의 정보 조건을 구분합니다. Reference registry 크기와 실제 native ensemble fitting label 수는 같은 개념이 아닙니다. 아래64-cell replay는 이미 저장된 모델을 읽으며 새 학습을 하지 않습니다. LiverCT는 원 query cohort에서20epoch 적응한 모델의 재추론으로, zero-adaptation 또는 새로운 strict-inductive 평가가 아닙니다.


In [9]:
for regime in ['inductive','query_adaptive']:
    table=pd.read_csv(REPORT/f'leaderboard_annotation_{regime}.tsv',sep='	')
    display(table[['model','feature_dimension','pooled_macro_f1','prediction_coverage','unlabeled_query_fit']])
reference=json.loads((REPORT/'model_parameter_audit/liverct.json').read_text())['records'][0]
print('Reference registry cells:',reference['reference_registry_n_cells'],'batches:',reference['reference_registry_n_batches'],'label categories:',reference['reference_label_categories'])
print(reference['reference_native_head_training_label_count'])
for model in ['scvi','liverct']:
    probe_id='classification_native64_v1' if model=='scvi' else 'classification_native64_v2'
    probe=json.loads((REPORT/f'annotation_native_inference_probe/{probe_id}_{model}.json').read_text())
    print(model,probe['n_real_query_cells'],probe['regime'],probe['comparisons'])


,model,feature_dimension,pooled_macro_f1,prediction_coverage,unlabeled_query_fit
0,pca,50.0,0.934829,1.0,False
1,scgpt,512.0,0.937697,1.0,False
2,geneformer,768.0,0.947083,1.0,False
3,geneformer_clcancer,768.0,0.945840,1.0,False
4,scvi,50.0,0.937858,1.0,False


,model,feature_dimension,pooled_macro_f1,prediction_coverage,unlabeled_query_fit
0,LiverCT_native,30.0,0.901671,0.887096,True
1,LiverCT_SCANVI_control,30.0,0.706087,0.803176,True


Reference registry cells: 331127 batches: 21 label categories: ['B cell', 'Cholangiocyte', 'Endothelial cell', 'Hepatocyte', 'Mesenchymal cell', 'Myeloid cell', 'NK/T cell', 'Plasma B cell']
Exact ensemble fitting example count not retained in the inspected checkpoint; VAE reference registry size is separate
scvi 64 strict_inductive_saved_fold_model {'scVI_changed_predictions_vs_original': 0, 'scVI_probability_max_abs_difference_vs_original': 5.071864707106855e-07}
liverct 64 replay_of_saved_query_adapted_context_not_zero_adaptation {'LiverCT_changed_predictions_vs_original': 0, 'matched_SCANVI_changed_predictions_vs_original': 0, 'native_broad_classifier_type': 'VotingClassifier', 'native_broad_classifier_members': ['RandomForestClassifier', 'MLPClassifier', 'OneVsRestClassifier', 'XGBClassifier'], 'native_broad_classifier_sha256': '52fcb8ed5ddb41a14cde787e293e406b3d6af5f1b8b279fbe44e5f7781f5ff27', 'auxiliary_state_outputs': 'Broad lv1 path exactly matches LabelPredict first stage. Un

In [10]:
SUBMIT_NATIVE_REPLAY = False
if SUBMIT_NATIVE_REPLAY:
    from datetime import datetime, timezone
    probe_id='tutorial06_native64_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
    result=subprocess.run(['sbatch','--parsable',str(ROOT/'slurm/replay_annotation_models.sh'),probe_id],cwd=ROOT,check=True,text=True,capture_output=True)
    print('submitted:',result.stdout.strip(),'probe:',probe_id)
else:
    print('저장 결과 읽기 모드: 새64-cell 추론을 제출하지 않았습니다.')


저장 결과 읽기 모드: 새64-cell 추론을 제출하지 않았습니다.


## donor 단위 불확실성과 calibration

8명 donor를 5,000번 쌍대 재표집한 개발 결과입니다. 이미 확인한 개발 데이터의 기술적 요약이며 새로운 blind test가 아닙니다. 학습 모델을 고정한 CI라서 재학습 변동은 포함하지 않습니다. Brier/ECE는 실제 pooled 확률로 계산하며, native hard-label 결과에 확률을 만들어 넣지 않습니다.


In [11]:
for regime in ['inductive','query_adaptive']:
    print(regime)
    display(pd.read_csv(REPORT/f'annotation_uncertainty_{regime}/paired_donor_bootstrap.tsv',sep='\t'))
    display(pd.read_csv(REPORT/f'annotation_uncertainty_{regime}/pooled_metrics_and_calibration.tsv',sep='\t'))


inductive


,candidate,reference,pooled_macro_f1_difference,lower95,upper95,n_donors
0,scgpt,pca,0.002868,-0.002460,0.006533,8
1,geneformer,pca,0.012255,0.000099,0.022147,8
2,geneformer_clcancer,geneformer,-0.001243,-0.003672,0.001447,8
3,scvi,pca,0.003030,-0.008647,0.010831,8


,model,macro_f1,balanced_accuracy,n_cells,n_known,evaluation_coverage,prediction_coverage,n_unknown_truth,n_abstained,confidence_calibration,answered_only_macro_f1,brier_multiclass,ece_10_bins
0,pca,0.934829,0.946531,63036,63036,1.0,1.0,0,0,uncalibrated; no confidence threshold; ECE/Bri...,0.934829,0.078118,0.041750
1,scgpt,0.937697,0.945257,63036,63036,1.0,1.0,0,0,uncalibrated; no confidence threshold; ECE/Bri...,0.937697,0.070927,0.007114
2,geneformer,0.947083,0.946851,63036,63036,1.0,1.0,0,0,uncalibrated; no confidence threshold; ECE/Bri...,0.947083,0.064447,0.004197
3,geneformer_clcancer,0.945840,0.944824,63036,63036,1.0,1.0,0,0,uncalibrated; no confidence threshold; ECE/Bri...,0.945840,0.063396,0.004478
4,scvi,0.937858,0.939070,63036,63036,1.0,1.0,0,0,uncalibrated; no confidence threshold; ECE/Bri...,0.937858,0.080462,0.012459


query_adaptive


,candidate,reference,pooled_macro_f1_difference,lower95,upper95,n_donors
0,LiverCT_native,LiverCT_SCANVI_control,0.195584,0.140466,0.245449,8


,model,macro_f1,balanced_accuracy,n_cells,n_known,evaluation_coverage,prediction_coverage,n_unknown_truth,n_abstained,confidence_calibration,answered_only_macro_f1
0,LiverCT_native,0.901671,0.879932,63036,63036,1.0,0.887096,0,7117,uncalibrated; no confidence threshold; ECE/Bri...,0.945821
1,LiverCT_SCANVI_control,0.706087,0.779307,63036,63036,1.0,0.803176,0,12407,uncalibrated; no confidence threshold; ECE/Bri...,0.783205


## V3: 실제 조직 위치와 OOF 오류

원본 site는 canonical tissue_site로 변환되어 있습니다. 새 학습 없이 기존 전체 개발 OOF 예측을 연결했습니다. PVTT는2명/2samples, Lymph는1명/1sample이며 donor와 lineage support가 다르므로 조직 위치의 독립적 효과를 추정하지 않습니다. 같은 환자 안의 조직 비교와 donor×site×lineage×UMI 사분위 표에도 support를 함께 제공합니다.


In [12]:
site=json.loads((REPORT/'tissue_error_analysis/provenance.json').read_text())
print('source mapping:',site['metadata_mapping'])
display(pd.read_csv(REPORT/'tissue_error_analysis/pooled_by_tissue.tsv',sep='\t'))
subgroups=pd.read_csv(REPORT/'tissue_error_analysis/donor_tissue_lineage.tsv',sep='\t')
display(subgroups[(subgroups.model=='geneformer') & (subgroups.n_truth>=50)].sort_values('recall').head(10))

print(json.loads((REPORT/'tissue_error_analysis/metadata_schema_resolution.json').read_text()))
display(pd.read_csv(REPORT/'tissue_error_analysis/support_by_tissue_site.tsv',sep='\t'))
display(pd.read_csv(REPORT/'tissue_error_analysis/within_donor_tissue_comparisons.tsv',sep='\t').head(12))
context=json.loads((REPORT/'tissue_error_analysis/context_provenance.json').read_text())
print('UMI context analysis:',context['within_context_comparison'])
print('Support가 있는59개 context의 기술적 비교이며 구성 통제가 인과 원인을 증명하지 않습니다.')


source mapping: {'original_field': 'site', 'canonical_field': 'tissue_site', 'source_path': '/mnt/S1/sdata/pubdata/Virtual_Cell/GEO/GSE149614/GSE149614_HCC.metadata.updated.txt.gz', 'source_sha256': '4c547861aec44be917ed2340645d6b30009fff4c77853e878fd61c037bc6e8c8', 'n_exact_value_matches': 63036}


,model,tissue_site,n_donors,macro_f1,balanced_accuracy,n_cells,n_known,evaluation_coverage,prediction_coverage,n_unknown_truth,n_abstained,confidence_calibration,answered_only_macro_f1
0,pca,Lymph,1,0.927549,0.937066,2843,2843,1.0,1.0,0,0,uncalibrated; no confidence threshold; ECE/Bri...,0.927549
1,pca,Normal,7,0.940880,0.943176,26725,26725,1.0,1.0,0,0,uncalibrated; no confidence threshold; ECE/Bri...,0.940880
2,pca,PVTT,2,0.887409,0.858783,5971,5971,1.0,1.0,0,0,uncalibrated; no confidence threshold; ECE/Bri...,0.887409
3,pca,Tumor,8,0.919787,0.927219,27497,27497,1.0,1.0,0,0,uncalibrated; no confidence threshold; ECE/Bri...,0.919787
4,scgpt,Lymph,1,0.918036,0.922423,2843,2843,1.0,1.0,0,0,uncalibrated; no confidence threshold; ECE/Bri...,0.918036
5,scgpt,Normal,7,0.950900,0.945917,26725,26725,1.0,1.0,0,0,uncalibrated; no confidence threshold; ECE/Bri...,0.950900
6,scgpt,PVTT,2,0.893735,0.866089,5971,5971,1.0,1.0,0,0,uncalibrated; no confidence threshold; ECE/Bri...,0.893735
7,scgpt,Tumor,8,0.916575,0.926466,27497,27497,1.0,1.0,0,0,uncalibrated; no confidence threshold; ECE/Bri...,0.916575
8,geneformer,Lymph,1,0.942014,0.929193,2843,2843,1.0,1.0,0,0,uncalibrated; no confidence threshold; ECE/Bri...,0.942014
9,geneformer,Normal,7,0.953001,0.948213,26725,26725,1.0,1.0,0,0,uncalibrated; no confidence threshold; ECE/Bri...,0.953001


,model,patient_id,tissue_site,lineage,n_truth,n_predicted,true_positive,false_negative,false_positive,recall,precision,n_group
233,geneformer,HCC03,Tumor,T/NK,203,51,50,153,1,0.246305,0.980392,4825
300,geneformer,HCC08,Tumor,B,77,34,31,46,3,0.402597,0.911765,4833
219,geneformer,HCC01,Tumor,Hepatocyte,71,39,33,38,6,0.464789,0.846154,3368
299,geneformer,HCC08,PVTT,T/NK,348,202,190,158,12,0.545977,0.940594,4142
245,geneformer,HCC04,Tumor,T/NK,253,152,147,106,5,0.581028,0.967105,3501
273,geneformer,HCC07,Normal,Hepatocyte,53,33,32,21,1,0.603774,0.969697,3740
249,geneformer,HCC05,Normal,Hepatocyte,53,36,36,17,0,0.679245,1.000000,4656
315,geneformer,HCC10,Normal,Hepatocyte,99,82,68,31,14,0.686869,0.829268,3072
228,geneformer,HCC03,Tumor,B,71,74,52,19,22,0.732394,0.702703,4825
279,geneformer,HCC07,PVTT,Hepatocyte,150,124,113,37,11,0.753333,0.911290,1829


{'status': 'resolved', 'project_commit': '3c10a6588b49d340b3fcc2074a35fe473ea3da9b', 'canonical_field': 'tissue_site', 'selected_field': 'tissue_site', 'available_aliases': ['tissue_site'], 'original_field': 'site', 'mapping_source': 'scripts/01_prepare_hcc.py schema_mapping; original metadata value equality already checked in tissue_error_analysis/provenance.json', 'conflicts': [], 'no_inferred_labels': True}


,tissue_site,n_cells,n_donors,n_samples
0,Lymph,2843,1,1
1,Normal,26725,7,7
2,PVTT,5971,2,2
3,Tumor,27497,8,8


,model,patient_id,lineage,left_tissue,right_tissue,left_n_truth,right_n_truth,left_recall,right_recall,right_minus_left_recall
0,geneformer,HCC03,B,Normal,Tumor,121,71,0.983471,0.732394,-0.251077
1,geneformer,HCC03,Endothelial,Normal,Tumor,218,20,0.931193,0.550000,-0.381193
2,geneformer,HCC03,Fibroblast,Normal,Tumor,21,10,1.000000,0.600000,-0.400000
3,geneformer,HCC03,Hepatocyte,Normal,Tumor,827,3474,0.990326,0.993092,0.002765
4,geneformer,HCC03,Myeloid,Normal,Tumor,1000,1047,0.961000,0.961796,0.000796
5,geneformer,HCC03,T/NK,Normal,Tumor,414,203,0.857488,0.246305,-0.611183
6,geneformer,HCC04,B,Normal,Tumor,188,24,0.978723,0.125000,-0.853723
7,geneformer,HCC04,Endothelial,Normal,Tumor,477,30,0.997904,0.933333,-0.064570
8,geneformer,HCC04,Fibroblast,Normal,Tumor,68,90,0.764706,0.988889,0.224183
9,geneformer,HCC04,Hepatocyte,Normal,Tumor,326,2454,0.917178,0.996740,0.079562


UMI context analysis: {'context': 'donor×tissue_site×lineage', 'minimum_cells_per_extreme_quartile': 5, 'n_supported_contexts': 59, 'weight': 'Full-development population size of each supported context; fixed common weighting for Q1 and Q4', 'crude_Q4_minus_Q1': {'pca_correct': -0.042578843835268665, 'scgpt_correct': -0.05013008439621802, 'geneformer_correct': -0.046830382638492374, 'geneformer_clcancer_correct': -0.049241703153753424, 'scvi_correct': -0.03972333269877526, 'all_three_wrong': 0.038581128244177934}, 'weighted_supported_context_Q4_minus_Q1': {'pca_correct_Q4_minus_Q1': -0.2610366691552237, 'scgpt_correct_Q4_minus_Q1': -0.24071748128743353, 'geneformer_correct_Q4_minus_Q1': -0.26077220240266163, 'geneformer_clcancer_correct_Q4_minus_Q1': -0.24463969511040395, 'scvi_correct_Q4_minus_Q1': -0.23415307918832812, 'all_three_wrong_Q4_minus_Q1': 0.23164197418114107}}
Support가 있는59개 context의 기술적 비교이며 구성 통제가 인과 원인을 증명하지 않습니다.
